# Phase5: 発展手法（自動拡張・Mixup/CutMix）と転移学習

**この回で学ぶこと**
- **自動拡張**（TrivialAugment / RandAugment）: 拡張の選択をおまかせにする
- **Mixup / CutMix**: 画像を混ぜる強い拡張。ただし向き不向きがある
- **転移学習**: 事前学習済みモデルを使うと、拡張よりはるかに効く

> ⏱ 所要時間: QUICK=Trueで約15分（ゼロから学習5条件×150エポック＋転移学習6条件×15エポック、各1シード）。
> 転移学習の初回は ResNet18 の事前学習重み（約45MB）を自動ダウンロードします。

In [ ]:
import sys
from pathlib import Path

# 正しい環境（プロジェクトの .venv）で動いているか確認する
try:
    import torch  # noqa: F401
except ModuleNotFoundError:
    raise SystemExit(
        f"torch が見つかりません。今のPython: {sys.executable}\n"
        "→ カーネルを「Python (image_augment_lab)」（または プロジェクトの .venv/bin/python）に切り替えてください。"
    )

# src/ をimportできるようにする（notebooks/ からでもプロジェクト直下からでも動く）
sys.path.insert(0, str(next(p for p in [Path.cwd() / "src", Path.cwd().parent / "src"] if p.exists())))
from nb_utils import *

%matplotlib inline

In [ ]:
QUICK = True
SEEDS = [0] if QUICK else [0, 1, 2]
EPOCHS = ep(150)
from torchvision.transforms import v2
from torchvision import datasets
import torch, random

## 1. 自動拡張の見た目

| 手法 | 内容 |
|---|---|
| **TrivialAugment** | 毎回1つの操作を、ランダムな強さで適用。パラメータ調整が不要 |
| **RandAugment** | 操作をN個（ここでは2個）、強さMで適用。調整するのはN, Mだけ |

In [ ]:
from PIL import Image
random.seed(1); torch.manual_seed(1)
img = v2.Resize((224, 224))(Image.open(sorted((DATA / "train/ants").glob("*.jpg"))[3]).convert("RGB"))
show_images([v2.TrivialAugmentWide()(img) for _ in range(4)], [f"Trivial {i+1}" for i in range(4)], cols=4, size=2.6)
show_images([v2.RandAugment(num_ops=2, magnitude=9)(img) for _ in range(4)], [f"RandAug {i+1}" for i in range(4)], cols=4, size=2.6);

## 2. Mixup / CutMix の見た目

2枚の画像を混ぜ、**ラベルも混ぜた「ソフトラベル」**にします。

- **Mixup**: 画像をピクセルごとに重み付き平均
- **CutMix**: 画像の一部を別の画像に貼り替え、ラベルは面積比で混合

In [ ]:
ds = datasets.ImageFolder(DATA / "train", T.make_transform("cropflip"))
# アリ・ハチを交互に8枚そろえたバッチを作る
idx = [0, 130, 1, 131, 2, 132, 3, 133]
x = torch.stack([ds[i][0] for i in idx]); y = torch.tensor([ds[i][1] for i in idx])
print("クラス:", ds.classes)

torch.manual_seed(0)
for name, mixer in [("Mixup", v2.MixUp(num_classes=2)), ("CutMix", v2.CutMix(num_classes=2))]:
    xm, ym = mixer(x, y)
    show_images(list(xm[:4]), [f"{name} ラベル={[round(v,2) for v in ym[i].tolist()]}" for i in range(4)], cols=4, size=2.8)

## 3. ゼロから学習: 拡張の比較（150エポック）

基準を「切り抜き＋反転（`cropflip`）」にして、自動拡張と Mixup/CutMix を比べます。

In [ ]:
configs = [("cropflip", "none"), ("trivial", "none"), ("randaug", "none"), ("cropflip", "mixup"), ("cropflip", "cutmix")]
scratch = []
for aug, mix in configs:
    print(f"== aug={aug} mix={mix} ==")
    d, _ = run_seeds(SEEDS, aug=aug, mix=mix, epochs=EPOCHS)
    scratch.append(d)
scratch = pd.concat(scratch)
mean_table(scratch)

### 結果の読み方（MANUAL.mdの記載値・3シード平均, test）

拡張なし 0.701 / 切り抜き+反転 0.801 / TrivialAugment 0.792 / **RandAugment 0.844** / +Mixup 0.753 / +CutMix 0.714

- **RandAugment は手作りの組み合わせと同等以上**。迷ったら第一候補。
- **Mixup/CutMix は逆効果。** 小さなCNN・150エポックでは混ぜた画像を学習しきれない。
  Mixup は train 精度が 0.69 まで下がり、gap がほぼ0（=学習不足）。**強すぎる正則化。**

## 4. 転移学習（ImageNet事前学習のResNet18）

ImageNet（約100万枚）で学習済みのモデルを、アリ/ハチ用に**微調整**します。すでに「エッジ・模様・形」の見方を知っているため、少量データでも強い。
`--model resnet18` 相当は `model_name="resnet18"`、学習率は小さめ（1e-4）にします。

In [ ]:
tl_configs = [("none", "none"), ("cropflip", "none"), ("trivial", "none"), ("cropflip", "cutmix")]
transfer = []
for aug, mix in tl_configs:
    print(f"== ResNet18 aug={aug} mix={mix} ==")
    d, _ = run_seeds(SEEDS, aug=aug, mix=mix, epochs=ep(15), model_name="resnet18", lr=1e-4)
    transfer.append(d)
transfer = pd.concat(transfer)
mean_table(transfer)

### 極少量データ（クラスあたり10枚）での転移学習

In [ ]:
tiny = []
for aug in ["none", "cropflip"]:
    print(f"== ResNet18 n/class=10 aug={aug} ==")
    d, _ = run_seeds(SEEDS, aug=aug, epochs=ep(15), model_name="resnet18", lr=1e-4, n_per_class=10)
    tiny.append(d)
tiny = pd.concat(tiny)
mean_table(tiny)

In [ ]:
summary = pd.DataFrame({
    "ゼロから(拡張なし)":     [0.701],
    "ゼロから(RandAugment)": [scratch[scratch.aug == "randaug"].test_acc.mean()],
    "転移学習(拡張なし)":     [transfer[transfer.aug == "none"].test_acc.mean()],
    "転移学習(切り抜き+反転)": [transfer[transfer.aug == "cropflip"].test_acc.mean()],
}).T.rename(columns={0: "test精度"})
summary.round(3)   # ゼロから(拡張なし)の0.701はMANUAL.mdの記載値（Phase3、3シード平均）

### 結果の読み方（MANUAL.mdの記載値・3シード平均, test）

| データ | 条件 | test |
|---|---|---|
| 全243枚 | 拡張なし | 0.892 |
| 全243枚 | 切り抜き+反転 | **0.939** |
| 全243枚 | TrivialAugment / +CutMix | 0.905 / 0.913 |
| 10枚/クラス | 拡張なし → 切り抜き+反転 | 0.783 → 0.805 |

- **最大の改善要因は転移学習**: 拡張なしでも 0.701 → 0.892（+19pt）。10枚/クラスでも 0.64 → 0.78。
- 拡張はその**上積み**（+4.7pt）。
- 事前学習モデルでは、train 精度が1.000でも test は高い。**train が高い＝悪い、ではない。val/test で判断する。**

## 🧪 試してみよう
1. `RandAugment(num_ops=2, magnitude=9)` の `magnitude` を 3 / 9 / 15 に変えて、`T.AUG_SETS["randaug"]` を書き換えて比べよう。
2. Mixup を転移学習（ResNet18）に使うとどうなる？ ゼロから学習のときと違う？
3. 転移学習で `lr=1e-2` にすると何が起きる？（事前学習の重みが壊れやすい）
4. **自分の課題に当てはめるなら**、まず何をする？ → 下の「実践的な指針」で確認しよう。

## まとめ: 実践的な指針
1. **まず転移学習を検討する。** 少量データでは拡張よりはるかに効く。
2. 拡張は**切り抜き＋水平反転**から。
3. 迷ったら **RandAugment / TrivialAugment**。
4. 拡張は**1種類ずつ**試して効果を切り分ける。
5. 拡張ありは収束が遅い。**十分なエポック**で比べる。
6. **Mixup/CutMix は小さなモデルでは慎重に**（未学習になりやすい）。
7. 拡張は**データの代わりにならない**。極少量ならデータ追加。
8. 評価は**複数シード平均**で。0.03程度の差を過信しない。
9. 分割は拡張の前、拡張は学習データのみ、test は最終モデルで1回だけ。

詳細は `MANUAL.md` を参照。

🎉 全フェーズ完了。